In [1]:
import os
import torch
import numpy as np
from wm_gym.wm_gym_env import (
    MatrixGenerationArgs, VLMRewardArgs, seed_everything,
    OpenAIRewardModel, load_matrix_gym_pipe, matrixGym, export_to_video
)
from wm_gym.wm_gym_utils import create_temp_video, Image, Video

/home/andy/miniconda3/envs/matrix/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/andy/miniconda3/envs/matrix/lib/python3.10/site-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


In [2]:
openai_api_key = ""  # Replace with your actual OpenAI API key
skip_reward = True if openai_api_key == "" else False

matrix_gen_config = MatrixGenerationArgs(
    prompt="On a lush green meadow, a white car is driving. From an overhead panoramic shot, \
            this car is adorned with blue and red stripes on its body, and it has a black spoiler at the rear. \
            The camera follows the car as it moves through a field of golden wheat, surrounded by green grass and trees. \
            In the distance, a river and some hills can be seen, with a cloudless blue sky above.",
    model_path="/home/andy/matrix_stage4_ckpt",
    video_path="/home/andy/matrix/base_video.mp4",
)

vlm_rm_config = VLMRewardArgs(
    model="gpt-4o",
    api_key=openai_api_key,  # Replace with your actual API key if testing live
    reward_query = (
        "Given these consecutive images from a car racing game, analyze the moving direction and the locations of possible obstacles. "
        "And answer the question: do you think the car has any collision with obstacles during the process? "
        "In this game, collisions don’t actually cause any damage — even if the car passes through an obstacle, it still counts as a collision."
    ),
    reward_criteria = (
        "Here is the video description: {} "
        "Return 1 if there is not any collision happened between the car and an obstacle, "
        "Return -1 if you believe a collision has already occurred between the car and an obstacle, regardless of whether there was damage."
        "Your response must only contain one of the following: 1 or -1. Do not include any additional explanation or description."
    ),
)


In [3]:
seed_everything(matrix_gen_config.seed)
debug_clip_output_dir = "./debug_clip_output"
os.makedirs(debug_clip_output_dir, exist_ok=True)


In [4]:
gpt4_rm = OpenAIRewardModel(**vars(vlm_rm_config))
wm_gym_pipe = load_matrix_gym_pipe(matrix_gen_config, disable_progress_bar=True)
env = matrixGym(wm_gym_pipe, gpt4_rm)  # TODO: generate base video in this step

The config attributes {'invert_scale_latents': False} were passed to AutoencoderKLCogVideoX, but are not expected and will be ignored. Please verify your config.json configuration file.
Loading pipeline components...: 100%|██████████| 5/5 [00:01<00:00,  3.78it/s]


Latent size: torch.Size([1, 5, 16, 60, 90])


In [5]:
step_count = 0
state, info = env.reset()
video_clip = env.render(mode="pil")
video_path = create_temp_video(video_clip)
display(Video(video_path, embed=True))
# export_to_video(video_clip, fps=matrix_gen_config.fps, output_video_path=os.path.join(debug_clip_output_dir, "0_reset_video.mp4"))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [6]:
step_count += 1
action = "DR"
state, reward, terminated, truncated, info = env.step(action, pad_k_step=7, skip_reward=skip_reward)
video_clip = env.render(mode="pil")
video_path = create_temp_video(video_clip)
print("Full response:", info["full_analysis"])
print("Extracted reward:", reward)
display(Video(video_path, embed=True))
# export_to_video(video_clip, fps=matrix_gen_config.fps, output_video_path=os.path.join(debug_clip_output_dir, f"{step_count}_{action}_video.mp4"))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Full response: Based on the images, the car is moving forward through an open field with sparse vegetation. Here's the analysis:

1. **Direction**: The car is consistently moving forward in each image, as indicated by the position and angle of the car relative to the background.

2. **Obstacles**: The field appears mostly clear, with some small bushes and trees scattered around. These are potential obstacles.

3. **Collision Analysis**: 
   - In the sequence of images, the car seems to be navigating through the open areas between the bushes and trees.
   - There are no visible obstacles directly in the car's path in any of the images.

Based on this analysis, it seems unlikely that the car has any collision with obstacles during this sequence. The car appears to be moving through clear spaces without intersecting any visible obstacles.
Extracted reward: 1


In [7]:
step_count += 1
action = "DR"
state, reward, terminated, truncated, info = env.step(action, pad_k_step=7, skip_reward=skip_reward)
video_clip = env.render(mode="pil")
video_path = create_temp_video(video_clip)
print("Full response:", info["full_analysis"])
print("Extracted reward:", reward)
display(Video(video_path, embed=True))
# export_to_video(video_clip, fps=matrix_gen_config.fps, output_video_path=os.path.join(debug_clip_output_dir, f"{step_count}_{action}_video.mp4"))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Full response: Based on the images, the car is moving forward on a dirt path surrounded by grassy fields. Here's the analysis:

1. **Direction**: The car is moving slightly to the right as it progresses through the images.

2. **Obstacles**: The main potential obstacles are the trees and bushes visible in the distance. However, they appear to be off the path the car is taking.

3. **Collision**: Since the car is staying on the open path and not veering towards the trees or bushes, it seems unlikely that there is any collision with obstacles during this sequence.

Overall, the car appears to be navigating the terrain without hitting any obstacles.
Extracted reward: 1


In [8]:
step_count += 1
action = "DL"
state, reward, terminated, truncated, info = env.step(action, pad_k_step=7, skip_reward=skip_reward)
video_clip = env.render(mode="pil")
video_path = create_temp_video(video_clip)
print("Full response:", info["full_analysis"])
print("Extracted reward:", reward)
display(Video(video_path, embed=True))
# export_to_video(video_clip, fps=matrix_gen_config.fps, output_video_path=os.path.join(debug_clip_output_dir, f"{step_count}_{action}_video.mp4"))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Full response: Based on the images, the car is moving off-road through a grassy area. Here's the analysis:

1. **Direction**: The car is moving from right to left across the images, gradually turning left towards the road.

2. **Obstacles**: 
   - There are trees and bushes on both sides of the path.
   - The car seems to be heading towards the road, which is clear of obstacles.

3. **Collision Analysis**:
   - In the sequence, the car appears to be avoiding the trees and bushes.
   - The path seems clear, and the car is not shown directly intersecting with any obstacles.

Based on the images, it does not appear that the car has any collisions with obstacles during this sequence.
Extracted reward: 1


In [9]:
step_count += 1
action = "D"
state, reward, terminated, truncated, info = env.step(action, pad_k_step=7, skip_reward=skip_reward)
video_clip = env.render(mode="pil")
video_path = create_temp_video(video_clip)
print("Full response:", info["full_analysis"])
print("Extracted reward:", reward)
display(Video(video_path, embed=True))
# export_to_video(video_clip, fps=matrix_gen_config.fps, output_video_path=os.path.join(debug_clip_output_dir, f"{step_count}_{action}_video.mp4"))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Full response: In the images, the car is moving forward on a grassy area parallel to a road. Here's the analysis:

1. **Direction**: The car is moving slightly to the right, towards the road, as seen by its gradual shift in position from left to right across the images.

2. **Obstacles**: The main potential obstacles are the trees and bushes along the side of the road.

3. **Collision Analysis**:
   - **Image 1**: The car is positioned away from the trees.
   - **Image 2**: The car is still clear of the trees but moving closer.
   - **Image 3**: The car is approaching the edge of the grassy area, nearing the trees.
   - **Image 4**: The car appears to be very close to the trees, possibly touching or passing through the foliage.

Based on the sequence, it seems likely that the car may have had a collision with the trees or bushes by the fourth image. The car's trajectory suggests it is moving towards the obstacles, and by the last image, it appears to be in close proximity to them.
Extr

In [10]:
step_count += 1
action = "D"
state, reward, terminated, truncated, info = env.step(action, pad_k_step=7, skip_reward=skip_reward)
video_clip = env.render(mode="pil")
video_path = create_temp_video(video_clip)
print("Full response:", info["full_analysis"])
print("Extracted reward:", reward)
display(Video(video_path, embed=True))
# export_to_video(video_clip, fps=matrix_gen_config.fps, output_video_path=os.path.join(debug_clip_output_dir, f"{step_count}_{action}_video.mp4"))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Full response: In the images, the car is moving on a grassy area parallel to a road. Here's the analysis:

1. **Direction**: The car is moving forward, slightly to the right, as seen by the changing position relative to the road and trees.

2. **Obstacles**: The main obstacles are the trees and bushes along the side of the road.

3. **Collision Analysis**:
   - **Image 1**: The car is close to the trees but not in contact.
   - **Image 2**: The car maintains a similar distance from the trees.
   - **Image 3**: The car appears to be moving slightly closer to the trees.
   - **Image 4**: The car is very close to the trees, possibly touching or passing through the foliage.

Based on the images, it seems likely that the car may have had a collision with the trees in Image 4, as it appears to be very close or overlapping with the foliage.
Extracted reward: -1


In [11]:
step_count += 1
action = "DR"
state, reward, terminated, truncated, info = env.step(action, pad_k_step=7, skip_reward=skip_reward)
video_clip = env.render(mode="pil")
video_path = create_temp_video(video_clip)
print("Full response:", info["full_analysis"])
print("Extracted reward:", reward)
display(Video(video_path, embed=True))
# export_to_video(video_clip, fps=matrix_gen_config.fps, output_video_path=os.path.join(debug_clip_output_dir, f"{step_count}_{action}_video.mp4"))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Full response: In the sequence of images, the car appears to be moving along a road with a curve to the left. Here's the analysis:

1. **Image 1**: The car is on the grass, off the road, heading towards a tree and some bushes.
2. **Image 2**: The car is closer to the tree and bushes, indicating it is moving forward.
3. **Image 3**: The car is partially within the bushes, suggesting it has moved further into the obstacle area.
4. **Image 4**: The car is still in the bushes, indicating it has continued moving through the obstacle.

Based on this analysis, it seems the car has collided with the bushes and possibly the tree. The car's path suggests it has passed through these obstacles, which would count as collisions in the game.
Extracted reward: -1
